# 01 - Fuentes heterogéneas

En este notebook se integrarán distintas fuentes de datos del proyecto EdTech Learning Analytics.

Se trabajará con:
- archivos CSV con interacciones de estudiantes,
- una base de datos SQLite para estudiantes y cursos,
- y posteriormente una fuente JSON con indicadores derivados.

El objetivo es preparar una base integrada para el análisis de rendimiento académico y riesgo estudiantil.

## 1. Importación de librerías

Se cargan las librerías necesarias para manipulación de datos, conexión con SQLite, manejo de archivos JSON, rutas del proyecto y visualización básica.

## 2. Validación del entorno

Se verifica que el notebook esté utilizando correctamente las librerías instaladas en el entorno virtual del proyecto.

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import json
from pathlib import Path
import matplotlib.pyplot as plt

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("Entorno listo correctamente")

pandas: 3.1.0
numpy: 2.5.4
Entorno listo correctamente


## 3. Configuración de rutas

En esta sección se definen las rutas principales del proyecto.  
Esto permite acceder a los archivos de datos de forma ordenada y evitar escribir rutas absolutas en cada celda.

In [2]:
from pathlib import Path

# El notebook se encuentra dentro de /notebooks,
# por lo que la raíz del proyecto está un nivel arriba.
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "datos"
DATA_RAW = DATA_DIR / "crudos"
DATA_PROCESSED = DATA_DIR / "procesados"
DATA_EXTERNAL = DATA_DIR / "externos"

SQL_DIR = PROJECT_ROOT / "sql"
DOCUMENTS_DIR = PROJECT_ROOT / "documentos"
RESULTS_DIR = PROJECT_ROOT / "resultados"

print("Raíz del proyecto:", PROJECT_ROOT)
print("Datos crudos:", DATA_RAW)
print("Datos procesados:", DATA_PROCESSED)
print("Datos externos:", DATA_EXTERNAL)
print("SQL:", SQL_DIR)

Raíz del proyecto: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics
Datos crudos: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\crudos
Datos procesados: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\procesados
Datos externos: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\externos
SQL: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\sql


In [3]:
rutas = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "DATA_RAW": DATA_RAW,
    "DATA_PROCESSED": DATA_PROCESSED,
    "DATA_EXTERNAL": DATA_EXTERNAL,
    "SQL_DIR": SQL_DIR,
    "DOCUMENTS_DIR": DOCUMENTS_DIR,
    "RESULTS_DIR": RESULTS_DIR
}

for nombre, ruta in rutas.items():
    estado = "OK" if ruta.exists() else "NO EXISTE"
    print(f"{nombre:20} -> {estado} -> {ruta}")

PROJECT_ROOT         -> OK -> c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics
DATA_RAW             -> OK -> c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\crudos
DATA_PROCESSED       -> OK -> c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\procesados
DATA_EXTERNAL        -> OK -> c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\externos
SQL_DIR              -> OK -> c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\sql
DOCUMENTS_DIR        -> OK -> c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\documentos
RESULTS_DIR          -> OK -> c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\resultados


## 4. Carga inicial de los datasets

En esta sección se cargan los archivos CSV originales del proyecto.

El objetivo es verificar que los archivos puedan ser leídos correctamente y revisar de forma inicial su estructura, dimensiones y columnas disponibles, sin realizar modificaciones sobre los datos originales.

In [4]:
# Definición de rutas de los archivos fuente

ruta_students = DATA_RAW / "edtech_students.csv"
ruta_courses = DATA_RAW / "edtech_courses.csv"
ruta_interactions = DATA_RAW / "edtech_interactions.csv"

print("Students:", ruta_students)
print("Courses:", ruta_courses)
print("Interactions:", ruta_interactions)

Students: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\crudos\edtech_students.csv
Courses: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\crudos\edtech_courses.csv
Interactions: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\crudos\edtech_interactions.csv


In [5]:
archivos = {
    "edtech_students.csv": ruta_students,
    "edtech_courses.csv": ruta_courses,
    "edtech_interactions.csv": ruta_interactions
}

for nombre, ruta in archivos.items():
    estado = "OK" if ruta.exists() else "NO ENCONTRADO"
    print(f"{nombre:30} -> {estado}")

edtech_students.csv            -> OK
edtech_courses.csv             -> OK
edtech_interactions.csv        -> OK


In [6]:
# Carga de los datasets originales

df_students = pd.read_csv(ruta_students)
df_courses = pd.read_csv(ruta_courses)
df_interactions = pd.read_csv(ruta_interactions)

print("Datasets cargados correctamente")

Datasets cargados correctamente


In [7]:
print("Dimensiones de los datasets:")
print(f"Students:     {df_students.shape}")
print(f"Courses:      {df_courses.shape}")
print(f"Interactions: {df_interactions.shape}")

Dimensiones de los datasets:
Students:     (8000, 10)
Courses:      (150, 11)
Interactions: (200000, 12)


### 4.1 Vista preliminar de los datos

Se muestran los primeros registros de cada dataset para comprobar que las columnas y valores fueron interpretados correctamente por Pandas.

In [8]:
df_students.head()

,student_id,age,education_level,study_field,learning_style,device_preference,timezone,previous_online_courses,has_learning_disability,motivation_level
0,student_00000,46,Undergraduate,Social Sciences,Reading/Writing,Desktop,UTC-5,1,False,Medium
1,student_00001,16,High School,Social Sciences,Auditory,Mobile,UTC+8,6,True,Low
2,student_00002,27,Graduate,STEM,Reading/Writing,Mobile,UTC-5,7,False,High
3,student_00003,57,Undergraduate,Arts,Visual,Desktop,UTC-8,3,False,High
4,student_00004,39,High School,STEM,Auditory,Tablet,UTC,8,True,High


In [9]:
df_courses.head()

,course_id,course_title,subject_area,difficulty_level,duration_weeks,instructor_rating,course_format,has_assignments,has_quizzes,has_discussion_forum,certificate_offered
0,course_000,Economics 324,Art,Advanced,12,5.0,Interactive,True,True,True,False
1,course_001,History 563,Physics,Beginner,15,3.7,Video,False,True,True,True
2,course_002,History 244,English,Intermediate,5,4.6,Mixed,True,True,True,False
3,course_003,Computer Science 549,Physics,Intermediate,4,4.8,Interactive,True,True,True,True
4,course_004,Chemistry 574,English,Intermediate,9,3.6,Text-based,True,True,False,False


In [10]:
df_interactions.head()

,interaction_id,student_id,course_id,timestamp,activity_type,duration_seconds,completion_rate,score,attempts,device_used,help_sought,peer_interaction
0,interaction_000000,student_02020,course_078,2024-10-24 22:02:09,Note Taking,392,1.0,NaN,1,Tablet,False,False
1,interaction_000001,student_07410,course_067,2024-11-15 19:41:09,Assignment Submit,1683,1.0,0.9,2,Mobile,False,False
2,interaction_000002,student_03086,course_134,2024-06-29 21:33:09,Resource Download,1215,0.0,NaN,1,Desktop,False,False
3,interaction_000003,student_05092,course_112,2024-11-18 03:24:09,Note Taking,274,0.0,NaN,1,Desktop,False,False
4,interaction_000004,student_03626,course_011,2024-06-30 03:13:09,Note Taking,821,1.0,NaN,1,Mobile,False,False


### 4.2 Revisión de columnas

Se identifican las variables disponibles en cada fuente de datos. Esto permitirá determinar las llaves de integración y los atributos que podrán utilizarse posteriormente para el análisis.

In [11]:
print("Columnas de Students:")
print(df_students.columns.tolist())

print("\nColumnas de Courses:")
print(df_courses.columns.tolist())

print("\nColumnas de Interactions:")
print(df_interactions.columns.tolist())

Columnas de Students:
['student_id', 'age', 'education_level', 'study_field', 'learning_style', 'device_preference', 'timezone', 'previous_online_courses', 'has_learning_disability', 'motivation_level']

Columnas de Courses:
['course_id', 'course_title', 'subject_area', 'difficulty_level', 'duration_weeks', 'instructor_rating', 'course_format', 'has_assignments', 'has_quizzes', 'has_discussion_forum', 'certificate_offered']

Columnas de Interactions:
['interaction_id', 'student_id', 'course_id', 'timestamp', 'activity_type', 'duration_seconds', 'completion_rate', 'score', 'attempts', 'device_used', 'help_sought', 'peer_interaction']


## 5. Exploración inicial y calidad de los datos

En esta sección se realiza una revisión inicial de la calidad de los datasets.

Se validarán:
- tipos de datos,
- valores nulos,
- registros duplicados,
- unicidad de las llaves principales,
- y consistencia básica entre las fuentes.

El objetivo es identificar posibles problemas antes de realizar la integración de los datos.

### 5.1 Estructura y tipos de datos

Se revisa la estructura de cada dataset, incluyendo el número de registros, columnas, tipos de datos y cantidad de valores no nulos.

In [12]:
print("=== STUDENTS ===")
df_students.info()

print("\n=== COURSES ===")
df_courses.info()

print("\n=== INTERACTIONS ===")
df_interactions.info()

=== STUDENTS ===
<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   student_id               8000 non-null   str  
 1   age                      8000 non-null   int64
 2   education_level          8000 non-null   str  
 3   study_field              8000 non-null   str  
 4   learning_style           8000 non-null   str  
 5   device_preference        8000 non-null   str  
 6   timezone                 8000 non-null   str  
 7   previous_online_courses  8000 non-null   int64
 8   has_learning_disability  8000 non-null   bool 
 9   motivation_level         8000 non-null   str  
dtypes: bool(1), int64(2), str(7)
memory usage: 570.4 KB

=== COURSES ===
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -

In [13]:
print("Tipos de datos - Students")
display(df_students.dtypes.to_frame("dtype"))

print("\nTipos de datos - Courses")
display(df_courses.dtypes.to_frame("dtype"))

print("\nTipos de datos - Interactions")
display(df_interactions.dtypes.to_frame("dtype"))

Tipos de datos - Students


,dtype
student_id,str
age,int64
education_level,str
study_field,str
learning_style,str
device_preference,str
timezone,str
previous_online_courses,int64
has_learning_disability,bool
motivation_level,str



Tipos de datos - Courses


,dtype
course_id,str
course_title,str
subject_area,str
difficulty_level,str
duration_weeks,int64
instructor_rating,float64
course_format,str
has_assignments,bool
has_quizzes,bool
has_discussion_forum,bool



Tipos de datos - Interactions


,dtype
interaction_id,str
student_id,str
course_id,str
timestamp,str
activity_type,str
duration_seconds,int64
completion_rate,float64
score,float64
attempts,int64
device_used,str


### 5.2 Valores nulos

Se contabilizan los valores faltantes en cada variable para identificar campos que requieran tratamiento especial.

Un valor nulo no necesariamente representa un error; su interpretación dependerá del significado de la variable.

In [14]:
def resumen_nulos(df):
    resumen = pd.DataFrame({
        "valores_nulos": df.isnull().sum(),
        "porcentaje_nulos": (df.isnull().mean() * 100).round(2)
    })

    return resumen[resumen["valores_nulos"] > 0].sort_values(
        by="porcentaje_nulos",
        ascending=False
    )


print("Valores nulos - Students")
display(resumen_nulos(df_students))

print("Valores nulos - Courses")
display(resumen_nulos(df_courses))

print("Valores nulos - Interactions")
display(resumen_nulos(df_interactions))

Valores nulos - Students


,valores_nulos,porcentaje_nulos


Valores nulos - Courses


,valores_nulos,porcentaje_nulos


Valores nulos - Interactions


,valores_nulos,porcentaje_nulos
score,133419,66.71


### 5.3 Registros duplicados

Se verifica si existen filas completamente duplicadas dentro de cada dataset.

Los registros duplicados pueden afectar los análisis y producir resultados incorrectos si no son identificados previamente.

In [15]:
print("Duplicados completos:")
print(f"Students:     {df_students.duplicated().sum()}")
print(f"Courses:      {df_courses.duplicated().sum()}")
print(f"Interactions: {df_interactions.duplicated().sum()}")

Duplicados completos:
Students:     0
Courses:      0
Interactions: 0


### 5.4 Validación de llaves principales

Se valida que los identificadores principales sean únicos y que no contengan valores nulos.

Las llaves principales permiten identificar de forma única cada registro y serán utilizadas posteriormente para integrar las distintas fuentes.

In [16]:
def validar_llave(df, columna):
    print(f"Validación de {columna}")
    print(f"Registros totales:      {len(df)}")
    print(f"Valores únicos:         {df[columna].nunique()}")
    print(f"Valores duplicados:     {df[columna].duplicated().sum()}")
    print(f"Valores nulos:          {df[columna].isnull().sum()}")
    print()


validar_llave(df_students, "student_id")
validar_llave(df_courses, "course_id")
validar_llave(df_interactions, "interaction_id")

Validación de student_id
Registros totales:      8000
Valores únicos:         8000
Valores duplicados:     0
Valores nulos:          0

Validación de course_id
Registros totales:      150
Valores únicos:         150
Valores duplicados:     0
Valores nulos:          0

Validación de interaction_id
Registros totales:      200000


Valores únicos:         200000
Valores duplicados:     0
Valores nulos:          0



### 5.5 Consistencia entre fuentes

Se verifica que los identificadores de estudiantes y cursos presentes en las interacciones existan también en sus respectivos catálogos.

Esto permite validar la integridad referencial antes de realizar la integración de las fuentes.

In [17]:
students_interactions = set(df_interactions["student_id"].dropna().unique())
students_catalog = set(df_students["student_id"].dropna().unique())

courses_interactions = set(df_interactions["course_id"].dropna().unique())
courses_catalog = set(df_courses["course_id"].dropna().unique())


students_missing = students_interactions - students_catalog
courses_missing = courses_interactions - courses_catalog


print("=== Validación de estudiantes ===")
print(f"Estudiantes en interactions: {len(students_interactions)}")
print(f"Estudiantes en catálogo:     {len(students_catalog)}")
print(f"Estudiantes sin catálogo:    {len(students_missing)}")

print("\n=== Validación de cursos ===")
print(f"Cursos en interactions: {len(courses_interactions)}")
print(f"Cursos en catálogo:     {len(courses_catalog)}")
print(f"Cursos sin catálogo:    {len(courses_missing)}")

=== Validación de estudiantes ===
Estudiantes en interactions: 8000
Estudiantes en catálogo:     8000
Estudiantes sin catálogo:    0

=== Validación de cursos ===
Cursos en interactions: 150
Cursos en catálogo:     150
Cursos sin catálogo:    0


In [18]:
if len(students_missing) == 0:
    print("OK: Todos los estudiantes de interactions existen en students.")
else:
    print("Advertencia: existen estudiantes sin correspondencia.")

if len(courses_missing) == 0:
    print("OK: Todos los cursos de interactions existen en courses.")
else:
    print("Advertencia: existen cursos sin correspondencia.")

OK: Todos los estudiantes de interactions existen en students.
OK: Todos los cursos de interactions existen en courses.


### 5.6 Revisión de valores faltantes en score

La variable `score` contiene valores nulos.

Antes de considerarlos como errores, se analiza su distribución según el tipo de actividad, ya que algunas interacciones pueden no generar una calificación.

In [19]:
score_por_actividad = (
    df_interactions
    .groupby("activity_type")
    .agg(
        registros=("interaction_id", "count"),
        score_disponible=("score", "count"),
        score_nulo=("score", lambda x: x.isnull().sum())
    )
)

score_por_actividad["porcentaje_nulo"] = (
    score_por_actividad["score_nulo"]
    / score_por_actividad["registros"]
    * 100
).round(2)

display(score_por_actividad)

,registros,score_disponible,score_nulo,porcentaje_nulo
activity_type,,,,
Assignment Submit,33170,33170,0,0.0
Discussion Post,33328,0,33328,100.0
Note Taking,33754,0,33754,100.0
Quiz Attempt,33411,33411,0,0.0
Resource Download,33383,0,33383,100.0
Video Watch,32954,0,32954,100.0


## 6. Creación de la base de datos SQLite

En esta sección se crea una base de datos relacional SQLite para almacenar información maestra de estudiantes y cursos.

Se utilizarán dos tablas:

- `students`: catálogo de estudiantes.
- `courses`: catálogo de cursos.

Estas tablas permitirán integrar posteriormente la información relacional con el archivo de interacciones mediante `student_id` y `course_id`.

In [20]:
# Ruta de la base de datos SQLite

DB_PATH = DATA_RAW / "edtech_learning_analytics.db"

print("Base de datos:", DB_PATH)

Base de datos: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\crudos\edtech_learning_analytics.db


In [21]:
conexion = sqlite3.connect(DB_PATH)

print("Conexión SQLite creada correctamente")

Conexión SQLite creada correctamente


### 6.1 Creación de tablas

Se crean las tablas relacionales utilizando los identificadores principales como claves primarias.

In [22]:
cursor = conexion.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id TEXT PRIMARY KEY,
    age INTEGER,
    education_level TEXT,
    study_field TEXT,
    learning_style TEXT,
    device_preference TEXT,
    timezone TEXT,
    previous_online_courses INTEGER,
    has_learning_disability INTEGER,
    motivation_level TEXT
);
""")

conexion.commit()

print("Tabla students creada correctamente")

Tabla students creada correctamente


In [23]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS courses (
    course_id TEXT PRIMARY KEY,
    course_title TEXT,
    subject_area TEXT,
    difficulty_level TEXT,
    duration_weeks INTEGER,
    instructor_rating REAL,
    course_format TEXT,
    has_assignments INTEGER,
    has_quizzes INTEGER,
    has_discussion_forum INTEGER,
    certificate_offered INTEGER
);
""")

conexion.commit()

print("Tabla courses creada correctamente")

Tabla courses creada correctamente


### 6.2 Carga de datos en SQLite

Los datos originales de estudiantes y cursos se insertan en las tablas relacionales.

Se utiliza `to_sql` de Pandas para facilitar la carga de los registros.

In [24]:
students_sql = df_students.copy()
courses_sql = df_courses.copy()

In [25]:
conexion.execute("DELETE FROM students")
conexion.execute("DELETE FROM courses")
conexion.commit()

In [26]:
students_sql.to_sql(
    "students",
    conexion,
    if_exists="append",
    index=False
)

courses_sql.to_sql(
    "courses",
    conexion,
    if_exists="append",
    index=False
)

150

### 6.3 Validación de registros almacenados

Se consulta la base de datos para verificar que los registros hayan sido cargados correctamente.

In [27]:
cantidad_students = pd.read_sql_query(
    "SELECT COUNT(*) AS total FROM students;",
    conexion
)

cantidad_courses = pd.read_sql_query(
    "SELECT COUNT(*) AS total FROM courses;",
    conexion
)

print("Registros en students:")
display(cantidad_students)

print("Registros en courses:")
display(cantidad_courses)

Registros en students:


,total
0,8000


Registros en courses:


,total
0,150


### Resultado del bloque

La base de datos SQLite fue creada correctamente y contiene los catálogos de estudiantes y cursos.

Estas tablas funcionarán como fuentes relacionales y serán integradas posteriormente con el archivo `edtech_interactions.csv`.

In [28]:
df_students_sql = pd.read_sql_query(
    "SELECT * FROM students LIMIT 5;",
    conexion
)

df_courses_sql = pd.read_sql_query(
    "SELECT * FROM courses LIMIT 5;",
    conexion
)

print("Vista previa - Students")
display(df_students_sql)

print("Vista previa - Courses")
display(df_courses_sql)

Vista previa - Students


,student_id,age,education_level,study_field,learning_style,device_preference,timezone,previous_online_courses,has_learning_disability,motivation_level
0,student_00000,46,Undergraduate,Social Sciences,Reading/Writing,Desktop,UTC-5,1,0,Medium
1,student_00001,16,High School,Social Sciences,Auditory,Mobile,UTC+8,6,1,Low
2,student_00002,27,Graduate,STEM,Reading/Writing,Mobile,UTC-5,7,0,High
3,student_00003,57,Undergraduate,Arts,Visual,Desktop,UTC-8,3,0,High
4,student_00004,39,High School,STEM,Auditory,Tablet,UTC,8,1,High


Vista previa - Courses


,course_id,course_title,subject_area,difficulty_level,duration_weeks,instructor_rating,course_format,has_assignments,has_quizzes,has_discussion_forum,certificate_offered
0,course_000,Economics 324,Art,Advanced,12,5.0,Interactive,1,1,1,0
1,course_001,History 563,Physics,Beginner,15,3.7,Video,0,1,1,1
2,course_002,History 244,English,Intermediate,5,4.6,Mixed,1,1,1,0
3,course_003,Computer Science 549,Physics,Intermediate,4,4.8,Interactive,1,1,1,1
4,course_004,Chemistry 574,English,Intermediate,9,3.6,Text-based,1,1,0,0


## 7. Integración de fuentes de datos

En esta sección se integran las distintas fuentes del proyecto.

El archivo `edtech_interactions.csv` se utilizará como fuente principal y se complementará con la información almacenada en SQLite:

- `students`, mediante `student_id`.
- `courses`, mediante `course_id`.

El objetivo es construir un dataset unificado que concentre información de comportamiento, perfil del estudiante y características del curso.

In [29]:
# Lectura de tablas desde SQLite

df_students_sql = pd.read_sql_query(
    "SELECT * FROM students;",
    conexion
)

df_courses_sql = pd.read_sql_query(
    "SELECT * FROM courses;",
    conexion
)

print("Tablas leídas correctamente desde SQLite")

print(f"Students: {df_students_sql.shape}")
print(f"Courses:  {df_courses_sql.shape}")

Tablas leídas correctamente desde SQLite
Students: (8000, 10)
Courses:  (150, 11)


### 7.1 Integración de interacciones con estudiantes

Se realiza una unión de tipo `left join` entre las interacciones y el catálogo de estudiantes.

Se utiliza `student_id` como llave de relación.

El `left join` permite conservar todos los registros de interacciones, incluso si existiera algún estudiante sin correspondencia en el catálogo.

In [30]:
df_integrado = pd.merge(
    df_interactions,
    df_students_sql,
    on="student_id",
    how="left"
)

print("Integración con students completada")
print("Dimensiones:", df_integrado.shape)

Integración con students completada
Dimensiones: (200000, 21)


In [31]:
display(df_integrado.head())

,interaction_id,student_id,course_id,timestamp,activity_type,duration_seconds,completion_rate,score,attempts,device_used,...,peer_interaction,age,education_level,study_field,learning_style,device_preference,timezone,previous_online_courses,has_learning_disability,motivation_level
0,interaction_000000,student_02020,course_078,2024-10-24 22:02:09,Note Taking,392,1.0,NaN,1,Tablet,...,False,24,Undergraduate,Other,Reading/Writing,Desktop,UTC,5,0,Medium
1,interaction_000001,student_07410,course_067,2024-11-15 19:41:09,Assignment Submit,1683,1.0,0.9,2,Mobile,...,False,33,Professional,Arts,Auditory,Mobile,UTC-5,7,1,High
2,interaction_000002,student_03086,course_134,2024-06-29 21:33:09,Resource Download,1215,0.0,NaN,1,Desktop,...,False,27,Graduate,Other,Auditory,Mobile,UTC+1,7,0,Medium
3,interaction_000003,student_05092,course_112,2024-11-18 03:24:09,Note Taking,274,0.0,NaN,1,Desktop,...,False,44,Graduate,Other,Visual,Mobile,UTC-5,5,0,Medium
4,interaction_000004,student_03626,course_011,2024-06-30 03:13:09,Note Taking,821,1.0,NaN,1,Mobile,...,False,20,Graduate,Business,Kinesthetic,Desktop,UTC-8,6,1,High


### 7.2 Integración con cursos

Se integra el catálogo de cursos utilizando `course_id` como llave de relación.

Esto permite asociar cada interacción con información como el área temática, nivel de dificultad, modalidad y características del curso.

In [32]:
df_integrado = pd.merge(
    df_integrado,
    df_courses_sql,
    on="course_id",
    how="left"
)

print("Integración con courses completada")
print("Dimensiones finales:", df_integrado.shape)

Integración con courses completada
Dimensiones finales: (200000, 31)


In [33]:
display(df_integrado.head())

,interaction_id,student_id,course_id,timestamp,activity_type,duration_seconds,completion_rate,score,attempts,device_used,...,course_title,subject_area,difficulty_level,duration_weeks,instructor_rating,course_format,has_assignments,has_quizzes,has_discussion_forum,certificate_offered
0,interaction_000000,student_02020,course_078,2024-10-24 22:02:09,Note Taking,392,1.0,NaN,1,Tablet,...,Economics 384,Computer Science,Advanced,4,4.4,Video,1,1,0,1
1,interaction_000001,student_07410,course_067,2024-11-15 19:41:09,Assignment Submit,1683,1.0,0.9,2,Mobile,...,Music 579,English,Beginner,4,4.7,Mixed,1,1,1,1
2,interaction_000002,student_03086,course_134,2024-06-29 21:33:09,Resource Download,1215,0.0,NaN,1,Desktop,...,Language Learning 231,History,Beginner,7,3.9,Text-based,1,1,0,0
3,interaction_000003,student_05092,course_112,2024-11-18 03:24:09,Note Taking,274,0.0,NaN,1,Desktop,...,Biology 496,Art,Intermediate,5,3.8,Mixed,0,1,0,1
4,interaction_000004,student_03626,course_011,2024-06-30 03:13:09,Note Taking,821,1.0,NaN,1,Mobile,...,Psychology 251,Computer Science,Advanced,4,4.7,Video,1,1,1,1


### 7.3 Validación de la integración

Se verifica que el proceso de integración no haya provocado pérdida o duplicación inesperada de registros.

También se revisa que todos los estudiantes y cursos hayan encontrado una correspondencia válida.

In [34]:
print("Registros originales:")
print(len(df_interactions))

print("\nRegistros después de la integración:")
print(len(df_integrado))

print("\nDiferencia:")
print(len(df_integrado) - len(df_interactions))

Registros originales:
200000

Registros después de la integración:
200000

Diferencia:
0


In [35]:
print("Valores nulos después del merge:")

print(
    "Students sin correspondencia:",
    df_integrado["age"].isnull().sum()
)

print(
    "Courses sin correspondencia:",
    df_integrado["course_title"].isnull().sum()
)

Valores nulos después del merge:
Students sin correspondencia: 0
Courses sin correspondencia: 0


In [36]:
print("Interacciones únicas:")
print(df_integrado["interaction_id"].nunique())

print("\nRegistros totales:")
print(len(df_integrado))

Interacciones únicas:
200000

Registros totales:
200000


### 7.4 Revisión de la estructura integrada

Se revisan las columnas disponibles después de combinar las fuentes.

El dataset resultante contiene información de tres dimensiones principales:

- comportamiento del estudiante,
- características personales,
- características del curso.

In [37]:
print(f"Total de columnas: {len(df_integrado.columns)}")

for columna in df_integrado.columns:
    print(columna)

Total de columnas: 31
interaction_id
student_id
course_id
timestamp
activity_type
duration_seconds
completion_rate
score
attempts
device_used
help_sought
peer_interaction
age
education_level
study_field
learning_style
device_preference
timezone
previous_online_courses
has_learning_disability
motivation_level
course_title
subject_area
difficulty_level
duration_weeks
instructor_rating
course_format
has_assignments
has_quizzes
has_discussion_forum
certificate_offered


In [38]:
columnas_interaccion = [
    "interaction_id",
    "student_id",
    "course_id",
    "timestamp",
    "activity_type",
    "duration_seconds",
    "completion_rate",
    "score",
    "attempts",
    "device_used",
    "help_sought",
    "peer_interaction"
]

columnas_estudiante = [
    "age",
    "education_level",
    "study_field",
    "learning_style",
    "device_preference",
    "timezone",
    "previous_online_courses",
    "has_learning_disability",
    "motivation_level"
]

columnas_curso = [
    "course_title",
    "subject_area",
    "difficulty_level",
    "duration_weeks",
    "instructor_rating",
    "course_format",
    "has_assignments",
    "has_quizzes",
    "has_discussion_forum",
    "certificate_offered"
]

print("Variables de interacción:", len(columnas_interaccion))
print("Variables de estudiante:", len(columnas_estudiante))
print("Variables de curso:", len(columnas_curso))

Variables de interacción: 12
Variables de estudiante: 9
Variables de curso: 10


### 7.5 Almacenamiento del dataset integrado

El resultado de la integración se almacena en la carpeta de datos procesados.

Este archivo servirá como base para los análisis posteriores sin modificar los archivos originales.

In [39]:
ruta_integrado = DATA_PROCESSED / "edtech_integrado.csv"

df_integrado.to_csv(
    ruta_integrado,
    index=False,
    encoding="utf-8"
)

print("Dataset integrado guardado en:")
print(ruta_integrado)

Dataset integrado guardado en:
c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\procesados\edtech_integrado.csv


In [40]:
print(
    "Archivo creado correctamente"
    if ruta_integrado.exists()
    else "Error al crear el archivo"
)

Archivo creado correctamente


### Resultado del bloque

Las tres fuentes fueron integradas correctamente.

El dataset resultante mantiene los registros originales de interacción y los complementa con información del estudiante y del curso.

Este conjunto integrado será utilizado como base para construir indicadores de participación, rendimiento académico y riesgo estudiantil.

## 8. Creación de una fuente JSON de indicadores estudiantiles

En esta sección se construye una fuente semiestructurada en formato JSON a partir de las interacciones registradas en la plataforma.

El objetivo es generar indicadores agregados por estudiante relacionados con:

- nivel de participación,
- tiempo de actividad,
- rendimiento,
- búsqueda de ayuda,
- interacción con compañeros,
- y progreso académico.

Esta fuente JSON complementará la información proveniente del archivo CSV y de la base de datos SQLite.

### 8.1 Preparación de indicadores por estudiante

Se agrupan las interacciones por `student_id` para calcular métricas generales de comportamiento y desempeño.

In [41]:
# Construcción de indicadores agregados por estudiante

df_student_metrics = (
    df_integrado
    .groupby("student_id")
    .agg(
        total_interactions=("interaction_id", "count"),
        total_duration_seconds=("duration_seconds", "sum"),
        average_duration_seconds=("duration_seconds", "mean"),
        average_completion_rate=("completion_rate", "mean"),
        average_score=("score", "mean"),
        average_attempts=("attempts", "mean"),
        help_sought_count=("help_sought", "sum"),
        peer_interaction_count=("peer_interaction", "sum")
    )
    .reset_index()
)

display(df_student_metrics.head())

,student_id,total_interactions,total_duration_seconds,average_duration_seconds,average_completion_rate,average_score,average_attempts,help_sought_count,peer_interaction_count
0,student_00000,22,16185,735.681818,0.901364,0.837500,1.318182,3,2
1,student_00001,23,19307,839.434783,0.863478,0.793636,1.652174,1,0
2,student_00002,26,16066,617.923077,0.831923,0.861429,1.230769,4,0
3,student_00003,20,12435,621.750000,0.803500,0.790000,1.200000,2,2
4,student_00004,23,22509,978.652174,0.864348,0.801429,1.173913,4,2


### 8.2 Conversión de duración a minutos

Para facilitar la interpretación, el tiempo total de actividad se convierte de segundos a minutos.

In [42]:
df_student_metrics["total_duration_minutes"] = (
    df_student_metrics["total_duration_seconds"] / 60
).round(2)

df_student_metrics["average_duration_seconds"] = (
    df_student_metrics["average_duration_seconds"]
).round(2)

df_student_metrics["average_completion_rate"] = (
    df_student_metrics["average_completion_rate"]
).round(4)

df_student_metrics["average_score"] = (
    df_student_metrics["average_score"]
).round(4)

df_student_metrics["average_attempts"] = (
    df_student_metrics["average_attempts"]
).round(2)

display(df_student_metrics.head())

,student_id,total_interactions,total_duration_seconds,average_duration_seconds,average_completion_rate,average_score,average_attempts,help_sought_count,peer_interaction_count,total_duration_minutes
0,student_00000,22,16185,735.68,0.9014,0.8375,1.32,3,2,269.75
1,student_00001,23,19307,839.43,0.8635,0.7936,1.65,1,0,321.78
2,student_00002,26,16066,617.92,0.8319,0.8614,1.23,4,0,267.77
3,student_00003,20,12435,621.75,0.8035,0.7900,1.20,2,2,207.25
4,student_00004,23,22509,978.65,0.8643,0.8014,1.17,4,2,375.15


### 8.3 Identificación de actividades evaluables

Se contabilizan las actividades que generan una calificación, específicamente `Assignment Submit` y `Quiz Attempt`.

Esto permitirá diferenciar participación general de rendimiento académico.

In [43]:
df_evaluables = df_integrado[
    df_integrado["activity_type"].isin(
        ["Assignment Submit", "Quiz Attempt"]
    )
]

df_evaluated_count = (
    df_evaluables
    .groupby("student_id")
    .agg(
        evaluated_activities=("interaction_id", "count")
    )
    .reset_index()
)

df_student_metrics = pd.merge(
    df_student_metrics,
    df_evaluated_count,
    on="student_id",
    how="left"
)

df_student_metrics["evaluated_activities"] = (
    df_student_metrics["evaluated_activities"]
    .fillna(0)
    .astype(int)
)

display(df_student_metrics.head())

,student_id,total_interactions,total_duration_seconds,average_duration_seconds,average_completion_rate,average_score,average_attempts,help_sought_count,peer_interaction_count,total_duration_minutes,evaluated_activities
0,student_00000,22,16185,735.68,0.9014,0.8375,1.32,3,2,269.75,8
1,student_00001,23,19307,839.43,0.8635,0.7936,1.65,1,0,321.78,11
2,student_00002,26,16066,617.92,0.8319,0.8614,1.23,4,0,267.77,7
3,student_00003,20,12435,621.75,0.8035,0.7900,1.20,2,2,207.25,4
4,student_00004,23,22509,978.65,0.8643,0.8014,1.17,4,2,375.15,7


### 8.4 Generación del archivo JSON

Los indicadores calculados se exportan en formato JSON para representar una fuente semiestructurada de información estudiantil.

In [44]:
student_engagement_json = []

for _, row in df_student_metrics.iterrows():

    registro = {
        "student_id": row["student_id"],

        "activity": {
            "total_interactions": int(row["total_interactions"]),
            "total_duration_minutes": float(row["total_duration_minutes"]),
            "average_duration_seconds": float(row["average_duration_seconds"])
        },

        "performance": {
            "average_completion_rate": (
                None
                if pd.isna(row["average_completion_rate"])
                else float(row["average_completion_rate"])
            ),
            "average_score": (
                None
                if pd.isna(row["average_score"])
                else float(row["average_score"])
            ),
            "evaluated_activities": int(row["evaluated_activities"]),
            "average_attempts": float(row["average_attempts"])
        },

        "engagement": {
            "help_sought_count": int(row["help_sought_count"]),
            "peer_interaction_count": int(row["peer_interaction_count"])
        }
    }

    student_engagement_json.append(registro)

In [45]:
ruta_json = DATA_EXTERNAL / "student_engagement.json"

with open(ruta_json, "w", encoding="utf-8") as archivo:
    json.dump(
        student_engagement_json,
        archivo,
        ensure_ascii=False,
        indent=4
    )

print("Archivo JSON creado correctamente:")
print(ruta_json)

Archivo JSON creado correctamente:
c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\externos\student_engagement.json


In [46]:
print("Total de estudiantes en JSON:")
print(len(student_engagement_json))

print("\nPrimer registro:")
print(json.dumps(
    student_engagement_json[0],
    ensure_ascii=False,
    indent=4
))

Total de estudiantes en JSON:
8000

Primer registro:
{
    "student_id": "student_00000",
    "activity": {
        "total_interactions": 22,
        "total_duration_minutes": 269.75,
        "average_duration_seconds": 735.68
    },
    "performance": {
        "average_completion_rate": 0.9014,
        "average_score": 0.8375,
        "evaluated_activities": 8,
        "average_attempts": 1.32
    },
    "engagement": {
        "help_sought_count": 3,
        "peer_interaction_count": 2
    }
}


### 8.5 Lectura y validación del JSON

Se vuelve a leer el archivo generado para validar que la estructura sea correcta y que pueda ser utilizada posteriormente como una fuente independiente.

In [47]:
with open(ruta_json, "r", encoding="utf-8") as archivo:
    datos_engagement = json.load(archivo)

print("Registros leídos desde JSON:")
print(len(datos_engagement))

print("\nEjemplo de registro:")
print(json.dumps(
    datos_engagement[0],
    ensure_ascii=False,
    indent=4
))

Registros leídos desde JSON:
8000

Ejemplo de registro:
{
    "student_id": "student_00000",
    "activity": {
        "total_interactions": 22,
        "total_duration_minutes": 269.75,
        "average_duration_seconds": 735.68
    },
    "performance": {
        "average_completion_rate": 0.9014,
        "average_score": 0.8375,
        "evaluated_activities": 8,
        "average_attempts": 1.32
    },
    "engagement": {
        "help_sought_count": 3,
        "peer_interaction_count": 2
    }
}


In [48]:
df_engagement_json = pd.json_normalize(datos_engagement)

display(df_engagement_json.head())

,student_id,activity.total_interactions,activity.total_duration_minutes,activity.average_duration_seconds,performance.average_completion_rate,performance.average_score,performance.evaluated_activities,performance.average_attempts,engagement.help_sought_count,engagement.peer_interaction_count
0,student_00000,22,269.75,735.68,0.9014,0.8375,8,1.32,3,2
1,student_00001,23,321.78,839.43,0.8635,0.7936,11,1.65,1,0
2,student_00002,26,267.77,617.92,0.8319,0.8614,7,1.23,4,0
3,student_00003,20,207.25,621.75,0.8035,0.7900,4,1.20,2,2
4,student_00004,23,375.15,978.65,0.8643,0.8014,7,1.17,4,2


### Resultado del bloque

Se generó correctamente una fuente semiestructurada en formato JSON con indicadores agregados por estudiante.

Con esto, el proyecto ya integra tres tipos de fuentes:

- archivo plano CSV,
- base de datos relacional SQLite,
- archivo semiestructurado JSON.

Estas fuentes podrán utilizarse posteriormente para analizar participación, rendimiento académico y riesgo estudiantil.

## 9. Integración del JSON y cálculo de indicadores iniciales

En esta sección se integra la fuente semiestructurada `student_engagement.json` con el dataset principal.

Posteriormente se calculan indicadores generales relacionados con participación, progreso y rendimiento académico.

El objetivo es obtener una primera visión analítica del comportamiento de los estudiantes antes de realizar análisis más avanzados.

### 9.1 Lectura de la fuente JSON

In [49]:
with open(ruta_json, "r", encoding="utf-8") as archivo:
    datos_engagement = json.load(archivo)

df_engagement = pd.json_normalize(datos_engagement)

print("Dimensiones del JSON convertido a DataFrame:")
print(df_engagement.shape)

display(df_engagement.head())

Dimensiones del JSON convertido a DataFrame:
(8000, 10)


,student_id,activity.total_interactions,activity.total_duration_minutes,activity.average_duration_seconds,performance.average_completion_rate,performance.average_score,performance.evaluated_activities,performance.average_attempts,engagement.help_sought_count,engagement.peer_interaction_count
0,student_00000,22,269.75,735.68,0.9014,0.8375,8,1.32,3,2
1,student_00001,23,321.78,839.43,0.8635,0.7936,11,1.65,1,0
2,student_00002,26,267.77,617.92,0.8319,0.8614,7,1.23,4,0
3,student_00003,20,207.25,621.75,0.8035,0.7900,4,1.20,2,2
4,student_00004,23,375.15,978.65,0.8643,0.8014,7,1.17,4,2


### 9.2 Preparación de nombres de columnas

Se simplifican los nombres generados a partir de la estructura JSON para facilitar su integración y análisis.

In [50]:
df_engagement = df_engagement.rename(columns={
    "activity.total_interactions": "total_interactions",
    "activity.total_duration_minutes": "total_duration_minutes",
    "activity.average_duration_seconds": "average_duration_seconds",
    "performance.average_completion_rate": "average_completion_rate",
    "performance.average_score": "average_score",
    "performance.evaluated_activities": "evaluated_activities",
    "performance.average_attempts": "average_attempts",
    "engagement.help_sought_count": "help_sought_count",
    "engagement.peer_interaction_count": "peer_interaction_count"
})

display(df_engagement.head())

,student_id,total_interactions,total_duration_minutes,average_duration_seconds,average_completion_rate,average_score,evaluated_activities,average_attempts,help_sought_count,peer_interaction_count
0,student_00000,22,269.75,735.68,0.9014,0.8375,8,1.32,3,2
1,student_00001,23,321.78,839.43,0.8635,0.7936,11,1.65,1,0
2,student_00002,26,267.77,617.92,0.8319,0.8614,7,1.23,4,0
3,student_00003,20,207.25,621.75,0.8035,0.7900,4,1.20,2,2
4,student_00004,23,375.15,978.65,0.8643,0.8014,7,1.17,4,2


### 9.3 Integración de indicadores con estudiantes

La información agregada del JSON se relaciona con el catálogo de estudiantes mediante `student_id`.

El resultado representa una vista analítica por estudiante.

In [51]:
df_student_analysis = pd.merge(
    df_students_sql,
    df_engagement,
    on="student_id",
    how="left"
)

print("Dimensiones del dataset analítico por estudiante:")
print(df_student_analysis.shape)

display(df_student_analysis.head())

Dimensiones del dataset analítico por estudiante:
(8000, 19)


,student_id,age,education_level,study_field,learning_style,device_preference,timezone,previous_online_courses,has_learning_disability,motivation_level,total_interactions,total_duration_minutes,average_duration_seconds,average_completion_rate,average_score,evaluated_activities,average_attempts,help_sought_count,peer_interaction_count
0,student_00000,46,Undergraduate,Social Sciences,Reading/Writing,Desktop,UTC-5,1,0,Medium,22,269.75,735.68,0.9014,0.8375,8,1.32,3,2
1,student_00001,16,High School,Social Sciences,Auditory,Mobile,UTC+8,6,1,Low,23,321.78,839.43,0.8635,0.7936,11,1.65,1,0
2,student_00002,27,Graduate,STEM,Reading/Writing,Mobile,UTC-5,7,0,High,26,267.77,617.92,0.8319,0.8614,7,1.23,4,0
3,student_00003,57,Undergraduate,Arts,Visual,Desktop,UTC-8,3,0,High,20,207.25,621.75,0.8035,0.7900,4,1.20,2,2
4,student_00004,39,High School,STEM,Auditory,Tablet,UTC,8,1,High,23,375.15,978.65,0.8643,0.8014,7,1.17,4,2


In [52]:
print("Registros de estudiantes:", len(df_student_analysis))
print("student_id únicos:", df_student_analysis["student_id"].nunique())

print(
    "Estudiantes sin métricas de engagement:",
    df_student_analysis["total_interactions"].isnull().sum()
)

Registros de estudiantes: 8000
student_id únicos: 8000
Estudiantes sin métricas de engagement: 0


### 9.4 Indicadores generales del proyecto

Se calculan métricas iniciales que permiten describir el comportamiento general de los estudiantes.

In [53]:
total_estudiantes = df_student_analysis["student_id"].nunique()

promedio_interacciones = (
    df_student_analysis["total_interactions"].mean()
)

promedio_tiempo_minutos = (
    df_student_analysis["total_duration_minutes"].mean()
)

promedio_completion = (
    df_student_analysis["average_completion_rate"].mean()
)

promedio_score = (
    df_student_analysis["average_score"].mean()
)

promedio_actividades_evaluadas = (
    df_student_analysis["evaluated_activities"].mean()
)

promedio_solicitudes_ayuda = (
    df_student_analysis["help_sought_count"].mean()
)

promedio_interacciones_pares = (
    df_student_analysis["peer_interaction_count"].mean()
)

In [54]:
print("=== Indicadores generales ===")

print(f"Total de estudiantes: {total_estudiantes}")
print(f"Promedio de interacciones: {promedio_interacciones:.2f}")
print(f"Promedio de tiempo activo: {promedio_tiempo_minutos:.2f} minutos")
print(f"Promedio de completion rate: {promedio_completion:.4f}")
print(f"Promedio de score: {promedio_score:.4f}")
print(f"Promedio de actividades evaluadas: {promedio_actividades_evaluadas:.2f}")
print(f"Promedio de solicitudes de ayuda: {promedio_solicitudes_ayuda:.2f}")
print(f"Promedio de interacciones con pares: {promedio_interacciones_pares:.2f}")

=== Indicadores generales ===
Total de estudiantes: 8000
Promedio de interacciones: 25.00
Promedio de tiempo activo: 300.77 minutos
Promedio de completion rate: 0.8111
Promedio de score: 0.8000
Promedio de actividades evaluadas: 8.32
Promedio de solicitudes de ayuda: 3.75
Promedio de interacciones con pares: 1.26


In [55]:
print(
    f"Promedio de completion rate: "
    f"{promedio_completion:.4f} "
    f"({promedio_completion * 100:.2f}%)"
)

print(
    f"Promedio de score: "
    f"{promedio_score:.4f} "
    f"({promedio_score * 100:.2f}%)"
)

Promedio de completion rate: 0.8111 (81.11%)
Promedio de score: 0.8000 (80.00%)


### 9.5 Rendimiento por nivel de motivación

Se analiza el rendimiento promedio de los estudiantes según su nivel de motivación.

In [56]:
rendimiento_por_motivacion = (
    df_student_analysis
    .groupby("motivation_level")
    .agg(
        estudiantes=("student_id", "count"),
        promedio_score=("average_score", "mean"),
        promedio_completion=("average_completion_rate", "mean"),
        promedio_interacciones=("total_interactions", "mean")
    )
    .round(4)
)

display(rendimiento_por_motivacion)

,estudiantes,promedio_score,promedio_completion,promedio_interacciones
motivation_level,,,,
High,2470,0.8010,0.8136,24.9955
Low,1596,0.7990,0.8135,24.9712
Medium,3934,0.7999,0.8086,25.0145


### 9.6 Rendimiento por nivel educativo

In [57]:
rendimiento_por_educacion = (
    df_student_analysis
    .groupby("education_level")
    .agg(
        estudiantes=("student_id", "count"),
        promedio_score=("average_score", "mean"),
        promedio_completion=("average_completion_rate", "mean"),
        promedio_interacciones=("total_interactions", "mean")
    )
    .round(4)
)

display(rendimiento_por_educacion)

,estudiantes,promedio_score,promedio_completion,promedio_interacciones
education_level,,,,
Graduate,1630,0.7996,0.8110,24.8847
High School,2408,0.8006,0.8147,24.8717
Professional,796,0.8000,0.8091,25.3442
Undergraduate,3166,0.7998,0.8090,25.0704


### 9.7 Comportamiento por estilo de aprendizaje

In [58]:
comportamiento_por_estilo = (
    df_student_analysis
    .groupby("learning_style")
    .agg(
        estudiantes=("student_id", "count"),
        promedio_score=("average_score", "mean"),
        promedio_completion=("average_completion_rate", "mean"),
        promedio_tiempo=("total_duration_minutes", "mean"),
        promedio_interacciones=("total_interactions", "mean")
    )
    .round(4)
)

display(comportamiento_por_estilo)

,estudiantes,promedio_score,promedio_completion,promedio_tiempo,promedio_interacciones
learning_style,,,,,
Auditory,2023,0.8001,0.8117,297.1636,24.8176
Kinesthetic,2011,0.8009,0.8103,300.9270,24.9910
Reading/Writing,2004,0.8002,0.8113,302.1960,25.0624
Visual,1962,0.7990,0.8111,302.8887,25.1335


### 9.8 Correlación inicial entre indicadores

Se calcula una matriz de correlación para identificar posibles relaciones lineales entre participación, progreso y rendimiento.

La correlación no implica causalidad, pero puede servir como punto de partida para análisis posteriores.

In [59]:
columnas_correlacion = [
    "total_interactions",
    "total_duration_minutes",
    "average_completion_rate",
    "average_score",
    "evaluated_activities",
    "average_attempts",
    "help_sought_count",
    "peer_interaction_count"
]

matriz_correlacion = (
    df_student_analysis[columnas_correlacion]
    .corr()
    .round(3)
)

display(matriz_correlacion)

,total_interactions,total_duration_minutes,average_completion_rate,average_score,evaluated_activities,average_attempts,help_sought_count,peer_interaction_count
total_interactions,1.000,0.740,0.016,0.009,0.572,0.004,0.379,0.216
total_duration_minutes,0.740,1.000,0.038,0.007,0.480,0.054,0.282,0.133
average_completion_rate,0.016,0.038,1.000,0.001,0.158,0.128,0.024,-0.007
average_score,0.009,0.007,0.001,1.000,-0.012,-0.004,0.000,0.014
evaluated_activities,0.572,0.480,0.158,-0.012,1.000,0.570,0.225,-0.006
average_attempts,0.004,0.054,0.128,-0.004,0.570,1.000,0.004,-0.102
help_sought_count,0.379,0.282,0.024,0.000,0.225,0.004,1.000,0.088
peer_interaction_count,0.216,0.133,-0.007,0.014,-0.006,-0.102,0.088,1.000


### 9.9 Identificación preliminar de estudiantes con bajo desempeño

Como primera aproximación, se identifican estudiantes cuyo rendimiento promedio se encuentra por debajo del promedio general.

Esta clasificación todavía no representa el indicador definitivo de riesgo estudiantil.

In [60]:
promedio_general_score = df_student_analysis["average_score"].mean()

df_student_analysis["below_average_score"] = (
    df_student_analysis["average_score"] < promedio_general_score
)

estudiantes_bajo_promedio = (
    df_student_analysis["below_average_score"].sum()
)

porcentaje_bajo_promedio = (
    estudiantes_bajo_promedio
    / total_estudiantes
    * 100
)

print(f"Score promedio general: {promedio_general_score:.4f}")
print(f"Estudiantes debajo del promedio: {estudiantes_bajo_promedio}")
print(f"Porcentaje debajo del promedio: {porcentaje_bajo_promedio:.2f}%")

Score promedio general: 0.8000
Estudiantes debajo del promedio: 4021
Porcentaje debajo del promedio: 50.26%


### 9.10 Almacenamiento del dataset analítico

Se guarda la vista consolidada por estudiante para utilizarla en las siguientes etapas del proyecto.

In [61]:
ruta_student_analysis = (
    DATA_PROCESSED / "edtech_student_analysis.csv"
)

df_student_analysis.to_csv(
    ruta_student_analysis,
    index=False,
    encoding="utf-8"
)

print("Dataset analítico guardado en:")
print(ruta_student_analysis)

Dataset analítico guardado en:
c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\procesados\edtech_student_analysis.csv


In [62]:
print(
    "Archivo creado correctamente"
    if ruta_student_analysis.exists()
    else "Error al crear el archivo"
)

Archivo creado correctamente


### Resultado del bloque

Se integró correctamente la fuente JSON con la información relacional de estudiantes.

Además, se calcularon indicadores iniciales de participación, progreso y rendimiento académico.

Estos resultados representan una primera aproximación al análisis de Learning Analytics y servirán como base para la posterior construcción de un indicador de riesgo estudiantil.

## 10. Conclusiones

En este notebook se integraron distintas fuentes de datos del proyecto **EdTech Learning Analytics** utilizando formatos y estructuras heterogéneas.

Se trabajó con:

- un archivo CSV de interacciones estudiantiles,
- una base de datos SQLite con información de estudiantes y cursos,
- y una fuente JSON con indicadores agregados de participación y rendimiento.

La integración permitió construir una vista unificada con información de comportamiento, perfil estudiantil y características de los cursos.

También se validó la calidad inicial de los datos mediante la revisión de valores nulos, registros duplicados, unicidad de identificadores e integridad referencial entre las fuentes.

Se identificó que los valores nulos de `score` corresponden a actividades que no generan calificación, por lo que no deben interpretarse como errores ni sustituirse automáticamente por cero.

Los indicadores iniciales muestran un promedio de rendimiento cercano al 80% y un nivel de finalización cercano al 81%.

La matriz de correlación evidencia relaciones importantes entre variables de participación, como el número de interacciones, el tiempo total de actividad y la cantidad de actividades evaluadas. Sin embargo, el rendimiento promedio (`average_score`) presenta correlaciones lineales muy bajas con las variables de participación analizadas.

Estos resultados indican que el riesgo estudiantil no debe construirse utilizando únicamente una variable o una relación lineal simple. Será necesario realizar análisis adicionales de distribución, segmentación y comportamiento antes de establecer criterios de riesgo.

El dataset integrado y la vista analítica por estudiante quedan preparados para las siguientes fases del proyecto.

In [63]:
print("=== Archivos generados ===")

archivos_generados = {
    "Base SQLite": DB_PATH,
    "Dataset integrado": ruta_integrado,
    "JSON de engagement": ruta_json,
    "Dataset analítico por estudiante": ruta_student_analysis
}

for nombre, ruta in archivos_generados.items():
    estado = "OK" if ruta.exists() else "NO ENCONTRADO"
    print(f"{nombre:35} -> {estado}")

=== Archivos generados ===
Base SQLite                         -> OK
Dataset integrado                   -> OK
JSON de engagement                  -> OK
Dataset analítico por estudiante    -> OK


### Archivos resultantes

Los principales archivos generados durante esta etapa son:

- `edtech_learning_analytics.db`: base de datos relacional con los catálogos de estudiantes y cursos.
- `edtech_integrado.csv`: dataset integrado a nivel de interacción.
- `student_engagement.json`: fuente semiestructurada con métricas agregadas por estudiante.
- `edtech_student_analysis.csv`: vista analítica consolidada a nivel de estudiante.

Los archivos originales permanecen sin modificaciones dentro de la carpeta de datos crudos.

## 10. Interpretación de inconsistencias detectadas

Durante las validaciones de consistencia se identificaron algunos casos donde la actividad registrada no coincide con las características declaradas del curso:

- existen registros `Quiz Attempt` en cursos con `has_quizzes = 0`,
- existen registros `Assignment Submit` en cursos con `has_assignments = 0`,
- y existen registros `Discussion Post` en cursos con `has_discussion_forum = 0`.

Estas situaciones no serán corregidas automáticamente, ya que pueden deberse a diferencias entre la configuración del curso y el comportamiento registrado en la plataforma, o a inconsistencias propias de la fuente original.

También se observó que el dispositivo utilizado coincide con la preferencia del estudiante en aproximadamente 42.31% de las interacciones. Esta diferencia no se considera un problema de calidad, ya que la preferencia de dispositivo no implica que el estudiante utilice siempre el mismo medio de acceso.

Por lo tanto, estas observaciones se documentan como inconsistencias semánticas o diferencias de comportamiento, pero los registros se conservan para no perder información potencialmente relevante.